<a href="https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap7_Payment_Sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Behavioral Scoring: Application Data and Payment Histories

*Deep Learning in Banking* (Bravo, Maldonado and Óskarsdóttir, Wiley, 2026), Chapter 7: updated case study (2026).

> **New case (2026).** This case replaces the case study printed in Chapter 7 of the book. The original case, which combines time series, static variables, images and text in one model, is still available: [open it in Colab](https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap7_Multimodal_Data.ipynb) or [view it on GitHub](https://github.com/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap7_Multimodal_Data.ipynb).

A behavioral scoring model ranks the loans a bank already holds by their risk of a serious delinquency. Banks build it from **hand-made features** of the payment history: the worst delinquency in the last 12 months, the number of late months, how much of the balance was paid down. Someone had to decide which features to compute.

In this lab the payment history is a **second modality**. Each loan has two kinds of data:

1. a **table**: the application data, one row per loan (credit score, loan-to-value, debt-to-income, and so on);
2. a **sequence**: the last 24 monthly payment records, six numbers per month.

We ask one question: **can the raw monthly records go into the model directly, without hand-made features, and do as well?** If they can, the gain is operational. Hand-made features have a technical debt: someone designs them, writes and validates the code, documents it, and keeps it working when the data change. A well-designed data flow, which turns the raw records into sequences, and a multimodal model that reads them, can remove that work.

We fuse the two modalities in the three ways of the chapter (early, late and a hybrid that we call residual fusion). We compare every fused model with each modality alone and with a strong gradient-boosting model that uses the hand-made features. A fused model must beat each modality alone. It needs to match the hand made features. We save all the technical debt of maintaining the model.

The data are half of the loans in the Freddie Mac Single-Family Loan-Level Dataset sample files for the origination years 1999 to 2007, the same data source we have been using on the book in a different period. We train on 2002 to 2005 and test on 2007, the year the US mortgage crisis started.

This is a **behavioural scoring model**: it ranks loans. We do not calibrate its outputs, so they should not be interpreted as probabilities of default (PDs).

**Before you start:** select a GPU runtime (*Runtime → Change runtime type → T4 GPU*).

Freddie Mac data is used with permission for academic purposes. Do not upload bank or customer data to Colab.

## 0. Setup

Colab already has every package we need: pandas, LightGBM, PyTorch and scikit-learn. We fix the random seed, and we check that the GPU is there.

In [ ]:
import hashlib
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
THREADS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("CPU cores:", THREADS, "| PyTorch", torch.__version__, "| LightGBM", lgb.__version__)
T_START = time.time()

## 1. The data: one table and one file of monthly records

We download two files:

- `lab_orig.parquet`: one row per loan, with the data known when the loan was granted;
- `lab_perf.parquet`: one row per loan and month, from April 2000 to December 2008, with the monthly record that the servicer reports.

The files keep Freddie Mac's raw codes. We clean them below. In this 2026 update, we are also using a more modern data format (parquet) instead of pandas, for efficiency.

In [ ]:
DATA = Path(os.environ.get("CH7_DATA", "data_ch7"))
FILES = {"lab_orig.parquet": "1yHBG4VmHiNlG_YpH0J63MmLyIW8pjUfj",   # Google Drive file ids
         "lab_perf.parquet": "1R6BlyHEn2RvA4LkvxMIrn83JbgMAI9bl"}
DATA.mkdir(exist_ok=True)
for name, file_id in FILES.items():
    if not (DATA / name).exists():
        import gdown
        gdown.download(id=file_id, output=str(DATA / name), quiet=False)

orig = pd.read_parquet(DATA / "lab_orig.parquet")
perf = pd.read_parquet(DATA / "lab_perf.parquet")
print(f"{len(orig):,} loans and {len(perf):,} monthly records")

Here is one loan's monthly record around its first late payment. `DQ_STATUS` counts the months the payment is late (00 is current, 03 is 90 days late; `RA` means the bank took the property). `CURRENT_UPB` is the unpaid balance. `MOD_FLAG` is `Y` in a month when the loan terms were modified and `P` in the months after. `ZERO_BALANCE_CODE` says why the balance went to zero (01 is a payoff; 02, 03, 09 and 15 end the loan after a credit problem).

In [ ]:
months_late = pd.to_numeric(perf.DQ_STATUS.astype(str), errors="coerce")
example = perf.loc[months_late >= 3, "LOAN_ID"].iloc[0]
rec = perf[perf.LOAN_ID == example].reset_index(drop=True)
first_late = (pd.to_numeric(rec.DQ_STATUS.astype(str), errors="coerce") >= 1).idxmax()
rec.iloc[max(0, first_late - 6): first_late + 8]

### Application data

Freddie Mac writes "not available" as a code: 9999 for a missing credit score, 999 for a missing ratio, 99 for a missing count. Used as numbers, these codes would look like very high values, so we turn them into missing values.

In [ ]:
MISSING = {"CREDIT_SCORE": 9999, "UNITS": 99, "CLTV": 999, "DTI": 999, "LTV": 999,
           "N_BORROWERS": 99, "MI_PCT": 999}
NUM = ["CREDIT_SCORE", "UNITS", "CLTV", "DTI", "ORIG_UPB", "LTV", "ORIG_RATE", "TERM", "N_BORROWERS", "MI_PCT"]
CAT = ["FIRST_TIME_HOMEBUYER", "OCCUPANCY", "CHANNEL", "PPM", "PROPERTY_TYPE", "PURPOSE"]

for col, code_value in MISSING.items():
    orig[col] = orig[col].where(orig[col] != code_value)
for col in CAT:
    orig[col] = orig[col].astype("category")
orig[NUM].describe().T.round(2)

## 2. From monthly records to sequences

We put the monthly records into one grid per variable: a row for each loan and a column for each month. Each grid holds one **channel** of the sequence:

| Channel | Meaning |
|---|---|
| `obs` | 1 if the loan has a record that month, 0 before origination |
| `dq` | months late, 0 to 6 (`RA` becomes 6) |
| `upb_ratio` | unpaid balance divided by the original balance |
| `rate_diff` | current interest rate minus the original rate |
| `mod` | 1 if the loan terms were modified this month or before |
| `defer` | 1 if part of the balance is deferred |

These are the raw records, rescaled so that loans of different sizes are comparable. Nothing is summarized: the network sees every month.

We also mark the months with a **default event**: 90 days late or more, the bank taking the property, or a credit-loss zero-balance code.

In [ ]:
def month_index(yyyymm):
    return (yyyymm // 100) * 12 + yyyymm % 100 - 1

M0 = month_index(200004)
N_MONTHS = month_index(200812) - M0 + 1
loans = pd.Index(orig.LOAN_ID)
r = loans.get_indexer(perf.LOAN_ID)                       # row of each record
m = month_index(perf.PERIOD.values) - M0                  # column of each record

dq_raw = perf.DQ_STATUS.astype(str).values
dq = pd.to_numeric(pd.Series(dq_raw), errors="coerce").values
ra = dq_raw == "RA"
zb = pd.to_numeric(perf.ZERO_BALANCE_CODE.astype(str), errors="coerce").values

CH = ["obs", "dq", "upb_ratio", "rate_diff", "mod", "defer"]
G = {c: np.full((len(loans), N_MONTHS), np.nan, np.float32) for c in CH}
G["obs"][r, m] = 1
G["dq"][r, m] = np.where(ra, 6, np.clip(dq, 0, 6))
G["upb_ratio"][r, m] = perf.CURRENT_UPB.values / orig.ORIG_UPB.values[r]
G["rate_diff"][r, m] = perf.CURRENT_RATE.values - orig.ORIG_RATE.values[r]
G["mod"][r, m] = np.isin(perf.MOD_FLAG.astype(str).values, ["Y", "P"])   # modified now or before
G["defer"][r, m] = perf.DEFERRED_UPB.fillna(0).values > 0
AGE = np.full((len(loans), N_MONTHS), np.nan, np.float32)
AGE[r, m] = perf.LOAN_AGE.astype(float).values
EVENT = np.zeros((len(loans), N_MONTHS), bool)
EVENT[r, m] = (dq >= 3) | ra | np.isin(zb, [2, 3, 9, 15])
print("grid per channel:", G["dq"].shape, "(loans x months)")

### Observation points, target and split

At the end of each quarter $t$ we score every loan that is **current** (not late) at $t$ and at least six months old. The input is the 24 months up to $t$. The target is 1 if a default event happens in the next 12 months.

The split is by time, so that every training label is known before the test period starts:

- **training**: quarter-ends from March 2002 to December 2005 (their 12-month windows close by December 2006);
- **test**: the four quarter-ends of 2007.

A loan appears once per quarter, so the same loan appears several times. Every split that holds data out (early stopping, folds, the test half) is therefore **by loan**, through a hash of the loan id, so that a loan's quarters never sit on both sides.

For the test we use only half of the loans. We designed this experiment on the other half of the 2007 loans, so this half was never looked at while we chose the models.

In [ ]:
L, H = 24, 12
TRAIN_T = [y * 100 + q for y in range(2002, 2006) for q in (3, 6, 9, 12)]
TEST_T = [200703, 200706, 200709, 200712]

def h01(ids, salt):
    """A number in [0, 1] from each loan id: the same id always gets the same number."""
    return np.array([int(hashlib.md5((salt + s).encode()).hexdigest()[:8], 16) / 0xFFFFFFFF for s in ids])

test_half = h01(loans, "half") >= 0.5
rows, seqs = [], []
for kind, quarters in (("train", TRAIN_T), ("test", TEST_T)):
    for t in quarters:
        k = month_index(t) - M0
        ok = (G["obs"][:, k] == 1) & (G["dq"][:, k] == 0) & (AGE[:, k] >= 6) & (G["upb_ratio"][:, k] > 0)
        if kind == "test":
            ok &= test_half
        i = np.where(ok)[0]
        seqs.append(np.stack([G[c][i, k - L + 1:k + 1] for c in CH], axis=2))
        rows.append(pd.DataFrame({"loan": i, "LOAN_ID": loans[i], "t": t, "kind": kind, "age": AGE[i, k],
                                  "y": EVENT[i, k + 1:k + 1 + H].any(axis=1).astype(int)}))
D = pd.concat(rows, ignore_index=True)
S = np.concatenate(seqs)                                  # rows x 24 months x 6 channels
D = D.join(orig[NUM + CAT].reset_index(drop=True), on="loan")

loan_es = h01(loans, "es") < 0.2                          # 20% of training loans for early stopping
loan_fold = (h01(loans, "fold") * 3).astype(int)          # 3 folds, for out-of-fold scores
train = (D.kind == "train").values
tr = train & ~loan_es[D.loan.values]
va = train & loan_es[D.loan.values]
te = (D.kind == "test").values
fold = loan_fold[D.loan.values]
for name, mask in (("training", tr), ("early stopping", va), ("test", te)):
    print(f"{name:15s} {mask.sum():8,} rows, {D.y[mask].sum():5,} events ({D.y[mask].mean():.2%})")

Here is the sequence of a test loan that defaulted, as the network sees it. The last month on the right is the quarter-end $t$, when the loan was current.

In [ ]:
j = np.where(te & (D.y.values == 1))[0][0]
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for a, c in zip(ax, ["dq", "upb_ratio", "rate_diff"]):
    a.plot(range(-L + 1, 1), S[j, :, CH.index(c)], marker="o")
    a.set_title(c); a.set_xlabel("months before t")
plt.tight_layout(); plt.show()

## 3. The hand-made features

This is an example of what a bank would compute from the same 24 months: the worst delinquency, the number of late months, the months since the last late payment, the balance paid down, and the current state of the loan. In reality, a behavioural scoring model may construct tens of thousands of variables (thus the technical debt!). A model with these features is the benchmark that the multimodal models must beat.

In [ ]:
def first_value(a):
    """The first non-missing value of each row."""
    ok = np.isfinite(a)
    v = a[np.arange(len(a)), ok.argmax(axis=1)]
    return np.where(ok.any(axis=1), v, np.nan)

def summaries(S):
    obs = np.nan_to_num(S[:, :, 0])
    dq = np.where(obs == 1, np.nan_to_num(S[:, :, 1]), 0)
    upb = S[:, :, 2]
    late12 = dq[:, -12:] >= 1
    f = pd.DataFrame(index=range(len(S)))
    f["max_dq12"] = dq[:, -12:].max(axis=1)
    f["n_dq12"] = late12.sum(axis=1)
    f["n_dq3"] = (dq[:, -3:] >= 1).sum(axis=1)
    f["since_dq"] = np.where(late12.any(axis=1), late12[:, ::-1].argmax(axis=1), 13)
    f["upb_ratio_t"] = upb[:, -1]
    f["paydown12"] = first_value(upb[:, -12:]) - upb[:, -1]
    f["rate_diff_t"] = S[:, -1, 3]
    f["mod_ever"] = np.nan_to_num(S[:, -12:, 4]).max(axis=1)
    f["defer_t"] = S[:, -1, 5]
    f["max_dq24"] = dq.max(axis=1)
    f["n_dq24"] = (dq >= 1).sum(axis=1)
    f["paydown24"] = first_value(upb) - upb[:, -1]
    return f

SUMM = summaries(S)
D = pd.concat([D, SUMM], axis=1)
TAB = NUM + CAT + ["age"]
SUMM_COLS = list(SUMM.columns)
D.loc[te, SUMM_COLS].describe().T.round(3)

## 4. Gradient boosting: one modality, then the hand-made features

We fit LightGBM twice: on the application data alone (**M1**), and with the hand-made features added (**M2**). M2 is the model a bank would build today. Early stopping uses the held-out 20% of training loans.

In [ ]:
RESULTS = {}          # model name -> test scores
INFO = {}             # model name -> inputs, fusion, seconds

def fit_lgb(Xtr, ytr, Xva, yva, seed=SEED):
    model = lgb.LGBMClassifier(n_estimators=4000, learning_rate=0.03, num_leaves=31, min_child_samples=200,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                               random_state=seed, n_jobs=THREADS, verbose=-1)
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
              callbacks=[lgb.early_stopping(200, verbose=False)])
    return model

def run_lgb(name, cols, inputs, fusion):
    t0 = time.time()
    model = fit_lgb(D.loc[tr, cols], D.y[tr], D.loc[va, cols], D.y[va])
    RESULTS[name] = model.predict_proba(D.loc[te, cols])[:, 1]
    INFO[name] = (inputs, fusion, time.time() - t0)
    print(f"{name}: test AUC {roc_auc_score(D.y[te], RESULTS[name]):.4f}, "
          f"{model.best_iteration_} trees, {time.time() - t0:.0f} s")
    return model

m1 = run_lgb("M1 LightGBM, application", TAB, "application", "none")
m2 = run_lgb("M2 LightGBM, application + hand-made", TAB + SUMM_COLS, "application + hand-made", "features")

## 5. Neural networks on the raw sequence

The networks need numbers without gaps on a common scale. For the table we turn each category into 0/1 columns, fill missing values with the training median (and add a 0/1 column that marks them), and standardize. For the sequence, a month before origination gets `obs` = 0 and the training median in the other channels; then each channel is standardized with one mean and one standard deviation, fitted on the training rows.

We move everything to the GPU once. The data are small enough to fit.

In [ ]:
def prep_tab(D, cols, fit):
    X = pd.get_dummies(D[cols], dtype=float)
    miss = X.isna().astype(float).add_suffix("_na")
    miss = miss.loc[:, miss[fit].sum() > 0]
    X = pd.concat([X.fillna(X[fit].median()), miss], axis=1)
    mu, sd = X[fit].mean(), X[fit].std().replace(0, 1)
    return ((X - mu) / sd).values.astype(np.float32)

def prep_seq(S, fit):
    S = S.copy()
    fill = np.nanmedian(S[fit].reshape(-1, S.shape[2]), axis=0)
    for c in range(S.shape[2]):
        v = S[:, :, c]
        v[np.isnan(v)] = 0.0 if c == 0 else fill[c]
    flat = S[fit].reshape(-1, S.shape[2])
    sd = flat.std(axis=0); sd[sd == 0] = 1
    return ((S - flat.mean(axis=0)) / sd).astype(np.float32)

gpu = lambda a: torch.tensor(a, dtype=torch.float32, device=DEVICE)
X_tab = gpu(prep_tab(D, TAB, tr))
X_seq = gpu(prep_seq(S, tr))
Y = D.y.values
ZERO = torch.zeros(len(D), device=DEVICE)
print("table:", tuple(X_tab.shape), "| sequence:", tuple(X_seq.shape))

One network class covers every design. It has up to two branches:

- a **table branch**, a small multilayer perceptron (MLP) on the application data;
- a **sequence branch**, a GRU that reads the 24 months in order and returns its last hidden state.

A **head** joins whatever branches are present and returns one number, the logit of the score. The head can also add a fixed **offset** to that logit. Residual fusion (Section 7) uses the offset.

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, n_tab, n_ch, use_tab=True, use_seq=True, zero_head=False):
        super().__init__()
        self.tab = nn.Sequential(nn.Linear(n_tab, 64), nn.ReLU(), nn.Dropout(0.1),
                                 nn.Linear(64, 32), nn.ReLU()) if use_tab else None
        self.seq = nn.GRU(n_ch, 64, batch_first=True) if use_seq else None
        width = (32 if use_tab else 0) + (64 if use_seq else 0)
        self.head = nn.Sequential(nn.Linear(width, 32), nn.ReLU(), nn.Linear(32, 1))
        if zero_head:                      # start as "no correction" (Section 7)
            nn.init.zeros_(self.head[2].weight)
            nn.init.zeros_(self.head[2].bias)

    def forward(self, x_tab, x_seq, offset):
        parts = []
        if self.tab is not None:
            parts.append(self.tab(x_tab))
        if self.seq is not None:
            parts.append(self.seq(x_seq)[1][-1])          # last hidden state
        return self.head(torch.cat(parts, dim=1)).squeeze(1) + offset

One training function serves every network. It trains with mini-batches, checks the AUC on the early-stopping loans after each epoch, keeps the best weights, and stops after three epochs without improvement. It returns the logits for the rows we ask for.

In [ ]:
def train_net(fit, stop, predict_for, use_tab=True, use_seq=True, x_seq=None, offset=None,
              zero_head=False, max_epochs=15, batch=2048, seed=SEED):
    set_seed(seed)
    x_seq = X_seq if x_seq is None else x_seq
    offset = ZERO if offset is None else offset
    net = FusionNet(X_tab.shape[1], x_seq.shape[2], use_tab, use_seq, zero_head).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    y = torch.tensor(Y, dtype=torch.float32, device=DEVICE)

    def logits(idx):
        net.eval(); out = []
        with torch.inference_mode():
            for b in range(0, len(idx), 16384):
                i = idx[b:b + 16384]
                out.append(net(X_tab[i], x_seq[i], offset[i]).float().cpu())
        net.train()
        return torch.cat(out).numpy()

    fit_i = torch.tensor(np.where(fit)[0], device=DEVICE)
    stop_i = torch.tensor(np.where(stop)[0], device=DEVICE)
    best, best_state, bad = -1.0, None, 0
    for epoch in range(1, max_epochs + 1):
        perm = fit_i[torch.randperm(len(fit_i), device=DEVICE)]
        for b in range(0, len(perm), batch):
            i = perm[b:b + batch]
            opt.zero_grad()
            loss_fn(net(X_tab[i], x_seq[i], offset[i]), y[i]).backward()
            opt.step()
        auc = roc_auc_score(Y[stop], logits(stop_i))
        if auc > best:
            best, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= 3:
                break
    net.load_state_dict(best_state)
    print(f"   {epoch} epochs, best early-stopping AUC {best:.4f}")
    return [logits(torch.tensor(np.where(p)[0], device=DEVICE)) for p in predict_for]

sigmoid = lambda z: 1 / (1 + np.exp(-z))

def record(name, logit_te, inputs, fusion, t0):
    RESULTS[name] = sigmoid(logit_te)
    INFO[name] = (inputs, fusion, time.time() - t0)
    print(f"{name}: test AUC {roc_auc_score(Y[te], RESULTS[name]):.4f}, {time.time() - t0:.0f} s")

### Each modality alone

**M3** is the MLP on the application data alone. **M4** is the GRU on the sequence alone. We train M4 three times, once per fold of training loans: each fold's network scores the loans it did not see. These **out-of-fold** scores are what late fusion needs in Section 7. M4's test score is the average of the three networks.

In [ ]:
t0 = time.time()
(lt,) = train_net(tr, va, [te], use_seq=False)
record("M3 MLP, application", lt, "application", "none", t0)

t0 = time.time()
gru_logit = np.full(len(D), np.nan)                       # out-of-fold on training rows
gru_rest = np.zeros(len(D))                               # fold average elsewhere
rest = ~tr
for f in range(3):
    hold, rest_logits = train_net(tr & (fold != f), va, [tr & (fold == f), rest], use_tab=False)
    gru_logit[tr & (fold == f)] = hold
    gru_rest[rest] += rest_logits / 3
gru_logit[rest] = gru_rest[rest]
record("M4 GRU, sequence", gru_logit[te], "raw sequence", "none", t0)

## 6. Early fusion

**M5** joins the two branches in one network and trains them together, so the GRU learns the sequence features that help *given* the application data. This is early (or intermediate) fusion: the modalities meet inside the model.

**M6** is a **placebo**. It is M5 trained and tested with the sequences shuffled across loans within each quarter. The sequences keep their distribution but no longer belong to their loans. If M6 does as well as M5, the sequence branch adds nothing real.

In [ ]:
t0 = time.time()
(lt,) = train_net(tr, va, [te])
record("M5 early fusion, MLP + GRU", lt, "application + raw sequence", "early", t0)

rng = np.random.default_rng(SEED)
perm = np.arange(len(D))
for t in D.t.unique():
    i = np.where(D.t.values == t)[0]
    perm[i] = rng.permutation(i)
t0 = time.time()
(lt,) = train_net(tr, va, [te], x_seq=X_seq[torch.tensor(perm, device=DEVICE)])
record("M6 placebo: M5, shuffled sequences", lt, "application + shuffled sequence", "early", t0)

## 7. Late fusion and residual fusion

In **late fusion** each modality has its own model, and a second model combines their scores. Here the GRU score of M4 becomes one more column for LightGBM:

- **M7**: LightGBM on the application data plus the GRU score. Raw sequence, no hand-made features.
- **M8**: LightGBM on the application data, the hand-made features and the GRU score. Does the network find anything that the hand-made features miss?

The GRU score of a training row must come from a network that did not train on that loan. Otherwise the score knows the label, and LightGBM trusts it too much. This is why M4 was trained out of fold.

In **residual fusion** gradient boosting goes first. LightGBM on the application data gives a logit; the network adds a **correction** to that logit, from both branches (**M9**). The head starts at zero, so the network starts as the LightGBM model and moves away only when the data say so. The LightGBM logit of a training row is again out of fold.

In [ ]:
D["gru_logit"] = gru_logit
m7 = run_lgb("M7 late fusion, LightGBM + GRU score", TAB + ["gru_logit"], "application + raw sequence", "late")
m8 = run_lgb("M8 late fusion, M2 + GRU score", TAB + SUMM_COLS + ["gru_logit"],
             "application + hand-made + raw sequence", "late")

t0 = time.time()
lgb_logit = np.zeros(len(D))
for f in range(3):
    fit_f, hold = tr & (fold != f), tr & (fold == f)
    model = fit_lgb(D.loc[fit_f, TAB], D.y[fit_f], D.loc[va, TAB], D.y[va])
    lgb_logit[hold] = model.predict(D.loc[hold, TAB], raw_score=True)
    lgb_logit[rest] += model.predict(D.loc[rest, TAB], raw_score=True) / 3
(lt,) = train_net(tr, va, [te], offset=gpu(lgb_logit), zero_head=True)
record("M9 residual fusion, LightGBM + network", lt, "application + raw sequence", "residual", t0)

## 8. Results

We report two measures on the test loans: the AUC, which measures the ranking, and the PR-AUC (average precision), which looks only at how well the riskiest scores catch the defaults. With under 1% events, the PR-AUC moves more than the AUC.

Differences between models come with a 95% interval from a **paired bootstrap by loan**: we draw loans with replacement, keep all their quarters, and recompute both models' AUC on the same draw. An interval that contains zero means the test data cannot tell the two models apart.

In [ ]:
def boot_diffs(y, scores, bench, groups, draws=200, seed=SEED):
    """Paired bootstrap by loan: AUC of each model minus AUC of the benchmark, on the same draws."""
    codes, uniq = pd.factorize(groups)
    rng = np.random.default_rng(seed)
    out = {name: [] for name in scores}
    for _ in range(draws):
        w = np.bincount(rng.integers(0, len(uniq), len(uniq)), minlength=len(uniq))[codes]
        base = roc_auc_score(y, scores[bench], sample_weight=w)
        for name, p in scores.items():
            out[name].append(roc_auc_score(y, p, sample_weight=w) - base)
    return {name: np.percentile(v, [2.5, 97.5]) for name, v in out.items()}

y_te, loans_te = Y[te], D.LOAN_ID.values[te]
BENCH = "M2 LightGBM, application + hand-made"
intervals = boot_diffs(y_te, RESULTS, BENCH, loans_te)
table = []
for name, p in RESULTS.items():
    inputs, fusion, secs = INFO[name]
    auc = roc_auc_score(y_te, p)
    row = {"model": name, "inputs": inputs, "fusion": fusion, "AUC": auc,
           "PR-AUC": average_precision_score(y_te, p), "seconds": secs}
    if name != BENCH:
        lo, hi = intervals[name]
        row["dAUC vs M2"] = auc - roc_auc_score(y_te, RESULTS[BENCH])
        row["95% interval"] = f"[{lo:+.4f}, {hi:+.4f}]"
    table.append(row)
table = pd.DataFrame(table).set_index("model")
print(f"Test: {te.sum():,} loan-quarters, {y_te.sum()} events, {len(set(loans_te)):,} loans")
table.round(4)

In [ ]:
order = table.index[::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(order, table.loc[order, "AUC"], color=["tab:gray" if "placebo" in n else "tab:blue" for n in order])
ax.axvline(table.loc[BENCH, "AUC"], color="tab:red", ls="--", label="M2, hand-made features")
ax.set_xlim(table.AUC.min() - 0.02, table.AUC.max() + 0.02); ax.set_xlabel("test AUC"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()
print(f"Total run time: {(time.time() - T_START) / 60:.1f} minutes")

### How to read the table

Check, in this order:

1. **Does the sequence add anything?** Compare M5 with M3 (the same network without the sequence) and M5 with M6 (the placebo). The sequence is useful only if M5 beats both.
2. **Do the raw records match the hand-made features?** Compare the raw-sequence models (M4, M5, M7, M9) with M2.
3. **Does the network find something the hand-made features miss?** Compare M8 with M2.
4. **Which fusion works?** Compare early (M5), late (M7) and residual (M9) fusion on the same inputs.

### What we found

We ran this notebook on a GPU of the size of a free Colab T4: 184,428 test loan-quarters with 1,487 events, about 4 minutes in all. Your numbers will differ a little, because training on a GPU is not exactly reproducible. The intervals are against M2.

1. **The sequence adds.** Early fusion (M5) reached an AUC of 0.839, against 0.789 for the same network without the sequence (M3). With shuffled sequences (M6) it fell to 0.794. The gain comes from the payment histories, not from the larger network.
2. **The raw records match the hand-made features.** M2 reached 0.841. Early fusion was −0.003 [−0.009, +0.004] from it, and residual fusion (M9) +0.004 [−0.002, +0.011]. Both intervals contain zero. The GRU alone (M4, 0.764) is far below: the sequence needs the application data next to it.
3. **The network finds little that the hand-made features miss.** Adding the GRU score to M2 (M8) gave +0.002 [−0.001, +0.005].
4. **Late fusion was the weakest of the three designs**: M7 was −0.005 [−0.007, −0.002] from M2.

The multimodal model does not beat the hand-made features, and it does not need to. It reaches the same ranking from the raw monthly records, with no feature design at all. The gain is operational: no feature library to design, validate, document and maintain. The work moves to the data flow of Section 2, which must build the sequences in the same way in training and in production.

## 9. The next step: transactional foundation models

The GRU in this lab learns its sequence features for one portfolio and one target. **Transactional foundation models** take the same idea further. A large transformer is pretrained with self-supervision (for example, by predicting the next event) on the raw event histories of millions of customers: card payments, transfers, app events. The bank then fine-tunes the same model for many tasks, such as scoring, fraud detection and product recommendation, and no task needs its own feature library.

- **E.T.-RNN** (Babaev et al., 2019) is an early step. A GRU on raw card transactions scored credit applications with a ROC AUC of 0.83, against 0.81 for LightGBM with about 7,000 hand-made aggregates (their Table 5).
- **nuFormer** (Braithwaite et al., 2025), at Nubank, pretrains a transformer on transaction sequences and fine-tunes it end to end together with the tabular features, as M5 does here. On credit default prediction it improves the AUC of the production LightGBM model, built on hand-made features, by 1.25% in relative terms. The transformer alone does worse than the tabular features alone. These are the same patterns as in this lab: a small gain in ranking, and a sequence model that needs the table next to it. The operational gain is larger. The authors report that they can remove 90% of the redundant hand-made transaction features with no loss in performance.
- **PRAGMA** (Ostroukhov et al., 2026), at Revolut, pretrains one transformer on banking event sequences and adapts it to credit scoring, fraud, lifetime value and other tasks. For credit scoring it reports ROC-AUC gains of 6% to 12% and PR-AUC gains of 70% to 130% over task-specific models, depending on model size (relative changes, their Table 2). The paper gives no absolute values, because they are commercially sensitive, and does not describe the baselines, so these gains cannot be compared with nuFormer's. Its operational argument is that one model can replace several high-maintenance task models.

The size of the accuracy gain from these models is therefore an open question. The operational argument, fewer features and fewer models to maintain, is the one that both papers make, and it is the one this lab shows.

## 10. Exercises

1. **Shorter history.** Set `L = 12` in Section 2 and run the lab again. Does the sequence still add? Do the hand-made features lose more than the networks?
2. **A transformer branch.** Replace the GRU with `nn.TransformerEncoder` (two layers, four heads, a learned position embedding), and pool the outputs over the months where `obs` is 1. Is it better than the GRU here?
3. **Which channels matter?** Train M5 with one channel at a time set to zero in `X_seq`. Which channel carries the signal?
4. **The crisis year.** The test year is 2007. Score the 2007 quarters one at a time. Do the models lose ranking power as the crisis deepens?
5. **From score to PD.** These are scores. What would you need to turn M2 or M9 into a PD model for a portfolio? Which data in this lab could you use to calibrate it, and which would you not trust?

## References

- Babaev, D., Savchenko, M., Tuzhilin, A. and Umerenkov, D. (2019). E.T.-RNN: Applying deep learning to credit loan applications. *KDD '19*, 2183–2190. https://doi.org/10.1145/3292500.3330693
- Braithwaite, D. T., Cavalcanti, M., McEver, R. A., Udagawa, H., Silva, D., Ramanath, R., Meneses, F., Yoshida, A., Wingert, E., Ramos, M., Zanfelice, B. and Gupta, A. (2025). Your spending needs attention: Modeling financial habits with transformers. arXiv:2507.23267.
- Bravo, C., Maldonado, S. and Óskarsdóttir, M. (2026). *Deep Learning in Banking*. Wiley.
- Cho, K., van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H. and Bengio, Y. (2014). Learning phrase representations using RNN encoder-decoder for statistical machine translation. *EMNLP 2014*, 1724–1734.
- Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q. and Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. *NeurIPS 2017*.
- Ostroukhov, M., Mikhailov, R., Iashin, V., Sokolov, A., Akshonov, A., Protasov, V., Goncharov, A., Beloborodov, D., Mullin, V., Enzmann, R. Y., Kolovos, G., Renders, J., Nesterov, P. and Repushko, A. (2026). PRAGMA: Revolut foundation model. arXiv:2604.08649.
- Freddie Mac. Single-Family Loan-Level Dataset. https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset